In [13]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, recall_score, roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from tqdm import tqdm
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report
import noisereduce as nr

from scipy.signal import butter, sosfilt
from scipy.stats import kurtosis, skew, entropy

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor

In [4]:
DCASE2024_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2024"

DCASE2024_TRAIN_PATH = DCASE2024_ROOT_PATH / "Train"
DCASE2024_DEV_PATH = DCASE2024_ROOT_PATH / "Dev"
DCASE2024_EVAL_PATH = DCASE2024_ROOT_PATH / "Eval"

DCASE2022_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2022"

DCASE2022_TRAIN_PATH = DCASE2022_ROOT_PATH / "Train"
DCASE2022_DEV_PATH = DCASE2022_ROOT_PATH / "Dev"

SAMPLE_RATE = 16_000

MEL_SPECTROGRAM_PARAMS = {
    "n_fft": 1024,
    "hop_length": 512,
    "n_mels": 128,
    "window": "hann",
    "center": True,
    "pad_mode": "reflect",
    "power": 2.0,
}

MFCC_PARAMS = {
    "n_mfcc": 20,
    "n_fft": 1024,
    "hop_length": 512,
    "n_mels": 128,
    "dct_type": 2,
    "norm": "ortho",
    "lifter": 0,
}

In [5]:
def load_audio(audio):
    signal, sample_rate = librosa.load(audio, mono=True, sr=SAMPLE_RATE)

    return signal, sample_rate

In [6]:
def pad_or_trim(signal, target_len):
    if len(signal) > target_len:
        return signal[:target_len]
    elif len(signal) < target_len:
        return np.pad(signal, (0, target_len - len(signal)), 'constant')
   
    return signal

def rms_normalize(signal, target_rms=0.1, eps=1e-8):
    rms = np.sqrt(np.mean(signal**2) + eps)
    gain = target_rms / (rms + eps)

    return (signal * gain).astype(np.float32)

def peak_normalize(signal):
    peak = np.max(np.abs(signal))
    if peak > 0:
        return signal / peak
    return signal

def bandpass_filter(signal, sr=16000, low=80, high=7500, order=4):
    nyq = 0.5 * sr
    low_n = low / nyq
    high_n = high / nyq
    sos = butter(order, [low_n, high_n], btype="bandpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

def spectral_gate(signal, sr=16000, noise_sec=0.25):
    n = int(noise_sec * sr)
    noise_clip = signal[:n]
    return nr.reduce_noise(y=signal, y_noise=noise_clip, sr=sr).astype(np.float32)

def highpass_filter(signal, sr=16000, cutoff=80, order=4):
    nyq = 0.5 * sr
    cutoff_n = cutoff / nyq
    sos = butter(order, cutoff_n, btype="highpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

In [10]:
def preprocess_data(signals, padding_or_trim=True, denoise_method=None, normalize_method=None):
    new_signals = []
    target_len = 10 * SAMPLE_RATE

    for signal in signals:
        if padding_or_trim:
            signal = pad_or_trim(signal, target_len)
        if denoise_method:
            signal = denoise_method(signal)
        if normalize_method:
            signal = normalize_method(signal)
        
        new_signals.append(signal)
        

    return np.array(signals)

def get_features_from_signal(signal, sample_rate):
    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=128)
    mfcc = np.mean(mfccs, axis=1)

    rms = np.sqrt(np.mean(signal**2))
    kurtosis_ = kurtosis(signal)
    skewness = skew(signal)
    peak_to_peak = np.ptp(signal)
    crest_factor = np.max(np.abs(y)) / rms_val if rms_val > 0 else 0
    mean = np.mean(signal)
    std = np.std(signal)

    y_abs = np.abs(y)
    y_prob = y_abs / np.sum(y_abs)
    entropy = entropy(y_prob)

    return np.numpy(mfcc), np.numpy([rms, kurtosis_, skewness, peak_to_peak, crest_factor, mean, std, entropy])

def get_features_from_signals(signals, sample_rate):
    features = []

    for signal in signals:
        mfcc, basic_features = get_features_from_signal(signal, sample_rate)
        features.append(basic_features)

    return np.array(features)

In [ ]:
def read_dcase_dev_set(path, ignore_machine_types=None, only_machine_type=None):
    if ignore_machine_types is None:
        ignore_machine_types = []

    X_train, y_train, mtype_train = [], [], []
    X_test, y_test, mtype_test = [], [], []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        mtype = machine_dir.name
        if mtype in ignore_machine_types:
            continue

        if only_machine_type is not None and mtype != only_machine_type:
            continue

        for section in ["train", "test"]:
            data_dir = machine_dir / section
            if not data_dir.exists():
                continue

            wavs = list(data_dir.glob("*.wav"))
            for audio_file in tqdm(wavs, desc=f"{mtype} - {section}"):
                try:
                    signal, _ = load_audio(str(audio_file))

                    if section == "train":
                        label = 0
                        X_train.append(signal)
                        y_train.append(label)
                        mtype_train.append(mtype)

                    else:
                        label = 1 if "anomaly" in audio_file.name.lower() else 0
                        X_test.append(signal)
                        y_test.append(label)
                        mtype_test.append(mtype)

                except Exception as e:
                    print(f"Erro ao processar {audio_file.name}: {e}")

    return X_train, X_test, y_train, y_test, mtype_train, mtype_test

In [ ]:
raw_signals_train, raw_signals_test, labels_train, labels_test, mtype_train, mtype_test = read_dcase_dev_set(DCASE2022_DEV_PATH, only_machine_type='fan')

print()
print('=' * 45)
print('Leitura Concluída!')
print(f'Total de amostras: {len(signals_train) + len(signals_test)}')
print(f'Amostras de treino: {len(signals_train)}')
print(f'Amostras de testes: {len(signals_test)}')
print(f'Máquinas de treino: {np.unique(mtype_train)}')
print(f'Máquinas de teste: {np.unique(mtype_test)}')
print('=' * 45)
print()

In [ ]:
signals_train = preprocess_data(raw_signals_train, 
                                padding_or_trim=True, 
                                denoise_method=bandpass_filter,
                                normalize_method=rms_normalize)

signals_test = preprocess_data(raw_signals_test, 
                                padding_or_trim=True, 
                                denoise_method=bandpass_filter,
                                normalize_method=rms_normalize)

X_train = get_features_from_signals(signals_train, SAMPLE_RATE)
X_test = get_features_from_signals(signals_test, SAMPLE_RATE)

std_scaler = StandardScaler()

std_scaler.fit(X_train)

X_train = std_scaler.transform(X_train)
X_test = std_scaler.transform(X_test)

In [ ]:
if_model = IsolationForest()

if_model.fit(signals_train)

if_scores = if_model.decision_function(signals_test)
if_auc = roc_auc_score(y_test, if_scores)
if_predict = if_model.predict(signals_test)

print(classification_report(y_test, if_predict))
print(confusion_matrix(y_test, if_predict))
print(f"AUC: {if_auc:.2%}")

In [ ]:
ocsvm_model = OneClassSVM()

ocsvm_model.fit(signals_train)

ocsvm_scores = ocsvm_model.decision_function(signals_test)
ocsvm_scores
